In [1]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEFECTS = ["defect_a", "defect_b", "defect_c", "defect_d"]
HEIGHT, WIDTH = 128, 800
BATCH_SIZE = 16


In [2]:
# Locate the mounted competition data without relying on one Kaggle mount layout.
known_roots = [
    Path("/kaggle/input/competitions/industrial-surface-defect-detection/public"),
    Path("/kaggle/input/industrial-surface-defect-detection/public"),
    Path("data/industrial-surface-defect-detection/public"),
]
ROOT = next((p for p in known_roots if (p / "train.csv").is_file()), None)
if ROOT is None:
    input_root = Path("/kaggle/input")
    matches = list(input_root.rglob("train.csv")) if input_root.exists() else []
    matches = [p.parent for p in matches if (p.parent / "sample_submission.csv").is_file()]
    if len(matches) != 1:
        raise FileNotFoundError(
            "Could not uniquely locate train.csv and sample_submission.csv under /kaggle/input. "
            f"Matching data directories: {matches}"
        )
    ROOT = matches[0]

TRAIN_CSV = ROOT / "train.csv"
SAMPLE_CSV = ROOT / "sample_submission.csv"
TRAIN_DIR = ROOT / "train"
print("Data root:", ROOT)
print("Train images directory exists:", TRAIN_DIR.is_dir())


Data root: /kaggle/input/competitions/industrial-surface-defect-detection/public
Train images directory exists: True


In [3]:
train_df = pd.read_csv(TRAIN_CSV, dtype={"image_id": str, "defect_type": str, "mask_rle": str})
sample_submission = pd.read_csv(
    SAMPLE_CSV, keep_default_na=False, dtype={"image_id": str, "defect_type": str, "mask_rle": str}
)
required_train = {"image_id", "defect_type", "mask_rle"}
required_submission = {"ID", "image_id", "defect_type", "mask_rle"}
if not required_train.issubset(train_df.columns):
    raise ValueError(f"train.csv must include {sorted(required_train)}; got {list(train_df.columns)}")
if not required_submission.issubset(sample_submission.columns):
    raise ValueError(
        f"sample_submission.csv must include {sorted(required_submission)}; "
        f"got {list(sample_submission.columns)}"
    )
if set(sample_submission["defect_type"].unique()) != set(DEFECTS):
    raise ValueError(
        "Update DEFECTS to exactly match the categories in sample_submission.csv: "
        f"{sorted(sample_submission['defect_type'].unique())}"
    )
if train_df.duplicated(["image_id", "defect_type"]).any():
    duplicates = train_df.loc[train_df.duplicated(["image_id", "defect_type"], keep=False),
                              ["image_id", "defect_type"]].head()
    raise ValueError(f"Expected at most one annotation per image/category; duplicates:\n{duplicates}")
unknown = set(train_df["defect_type"].dropna().unique()) - set(DEFECTS)
if unknown:
    raise ValueError(f"Unexpected defect categories in train.csv: {sorted(unknown)}")

# Keep blank or missing RLEs as empty masks.
label_table = (train_df.pivot(index="image_id", columns="defect_type", values="mask_rle")
               .reindex(columns=DEFECTS).fillna(""))
print("Train images:", len(label_table), "| annotations by category:")
print(label_table.apply(lambda col: col.map(lambda v: isinstance(v, str) and bool(v.strip())).sum()))
print("Submission rows:", len(sample_submission), "| columns:", list(sample_submission.columns))


Train images: 11393 | annotations by category:
defect_type
defect_a    1081
defect_b     313
defect_c    4100
defect_d    1707
dtype: int64
Submission rows: 12000 | columns: ['ID', 'image_id', 'defect_type', 'mask_rle']


In [4]:
def rle_decode(rle, shape=(HEIGHT, WIDTH)):
    flat = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    if rle is None or (isinstance(rle, float) and np.isnan(rle)) or not str(rle).strip():
        return flat.reshape(shape, order="F")
    values = np.asarray([int(v) for v in str(rle).split()], dtype=np.int64)
    if len(values) % 2:
        raise ValueError(f"RLE must contain start/length pairs: {rle!r}")
    starts = values[0::2] - 1
    lengths = values[1::2]
    if np.any(starts < 0) or np.any(lengths <= 0) or np.any(starts + lengths > flat.size):
        raise ValueError(f"RLE run falls outside mask of shape {shape}: {rle!r}")
    for start, length in zip(starts, lengths):
        flat[start:start + length] = 1
    return flat.reshape(shape, order="F")

def rle_encode(mask):
    pixels = np.asarray(mask, dtype=np.uint8).reshape(-1, order="F")
    padded = np.concatenate(([0], pixels, [0]))
    changes = np.flatnonzero(padded[1:] != padded[:-1])
    starts = changes[0::2] + 1
    lengths = changes[1::2] - changes[0::2]
    return " ".join(str(v) for pair in zip(starts, lengths) for v in pair)

def _has_mask(value):
    return isinstance(value, str) and bool(value.strip())

# Round-trip a bounded sample to catch layout or off-by-one mistakes before training.
nonempty_rles = train_df.loc[train_df["mask_rle"].map(_has_mask), "mask_rle"]
for rle in nonempty_rles.head(100):
    assert rle_encode(rle_decode(rle)) == rle.strip()
print(f"RLE round-trip passed for {min(100, len(nonempty_rles))} masks")


RLE round-trip passed for 100 masks


In [5]:
def index_images(directory):
    directory = Path(directory)
    if not directory.is_dir():
        raise FileNotFoundError(f"Image directory does not exist: {directory}")
    valid = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
    paths = {}
    for path in directory.rglob("*"):
        if path.is_file() and path.suffix.lower() in valid:
            key = path.stem
            if key in paths:
                raise ValueError(f"Duplicate image ID {key!r}: {paths[key]} and {path}")
            paths[key] = path
    return paths

train_paths = index_images(TRAIN_DIR)
missing_train = sorted(set(map(str, label_table.index)) - set(train_paths))
if missing_train:
    raise FileNotFoundError(f"Missing {len(missing_train)} training images; examples: {missing_train[:5]}")

# Split by image-level anomaly-presence pattern where the patterns have enough support.
defect_presence = label_table.apply(lambda col: col.map(_has_mask)).astype(np.uint8)
patterns = defect_presence.astype(str).agg("".join, axis=1)
pattern_counts = patterns.value_counts()
patterns = patterns.where(patterns.map(pattern_counts) >= 2, "rare")
val_size = max(1, int(round(0.15 * len(label_table))))
train_size = len(label_table) - val_size
stratify = (patterns if patterns.value_counts().min() >= 2
            and patterns.nunique() <= val_size and patterns.nunique() <= train_size else None)
train_ids, val_ids = train_test_split(
    label_table.index.to_numpy(), test_size=val_size, random_state=SEED, stratify=stratify
)
print(f"Train/validation images: {len(train_ids)}/{len(val_ids)}; stratified={stratify is not None}")


Train/validation images: 9684/1709; stratified=True


In [6]:
class StripDataset(Dataset):
    def __init__(self, ids, rle_table, paths, augment=False):
        self.ids = list(map(str, ids))
        self.rles = rle_table.copy()
        self.paths = paths
        self.augment = augment

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        image_id = self.ids[index]
        with Image.open(self.paths[image_id]) as image_file:
            image = np.asarray(image_file.convert("L"), dtype=np.float32) / 255.0
        if image.shape != (HEIGHT, WIDTH):
            raise ValueError(f"Expected image {image_id} to be {(HEIGHT, WIDTH)}, got {image.shape}")
        row = self.rles.loc[image_id]
        masks = np.stack([rle_decode(row.get(defect, "")) for defect in DEFECTS]).astype(np.float32)
        if self.augment:
            if random.random() < 0.5:
                image, masks = image[:, ::-1], masks[:, :, ::-1]
            if random.random() < 0.5:
                image, masks = image[::-1, :], masks[:, ::-1, :]
            image = np.clip(image * random.uniform(0.8, 1.2), 0.0, 1.0)
        image = np.ascontiguousarray(image[None, ...])
        masks = np.ascontiguousarray(masks)
        return torch.from_numpy(image), torch.from_numpy(masks)

train_ds = StripDataset(train_ids, label_table, train_paths, augment=True)
val_ds = StripDataset(val_ids, label_table, train_paths, augment=False)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 2
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                          pin_memory=(DEVICE.type == "cuda"))
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                        pin_memory=(DEVICE.type == "cuda"))
print("Device:", DEVICE, "| batch:", next(iter(train_loader))[0].shape)


Device: cpu | batch: torch.Size([16, 1, 128, 800])


In [7]:
def conv_block(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
    )

class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=len(DEFECTS), base=16):
        super().__init__()
        channels = [base, base * 2, base * 4, base * 8, base * 16]
        self.encoders = nn.ModuleList([
            conv_block(in_channels, channels[0]),
            conv_block(channels[0], channels[1]),
            conv_block(channels[1], channels[2]),
            conv_block(channels[2], channels[3]),
        ])
        self.bottleneck = conv_block(channels[3], channels[4])
        self.ups = nn.ModuleList([
            nn.ConvTranspose2d(channels[i + 1], channels[i], 2, stride=2)
            for i in range(3, -1, -1)
        ])
        self.decoders = nn.ModuleList([conv_block(channels[i] * 2, channels[i]) for i in range(3, -1, -1)])
        self.head = nn.Conv2d(channels[0], out_channels, 1)

    def forward(self, x):
        skips = []
        for encoder in self.encoders:
            x = encoder(x)
            skips.append(x)
            x = F.max_pool2d(x, 2)
        x = self.bottleneck(x)
        for up, decoder, skip in zip(self.ups, self.decoders, reversed(skips)):
            x = up(x)
            x = decoder(torch.cat((x, skip), dim=1))
        return self.head(x)

def segmentation_loss(logits, targets, eps=1.0):
    bce = F.binary_cross_entropy_with_logits(logits, targets)
    probabilities = torch.sigmoid(logits)
    intersection = (probabilities * targets).sum(dim=(2, 3))
    denominator = probabilities.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
    dice = 1.0 - (2.0 * intersection + eps) / (denominator + eps)
    return bce + dice.mean()

In [8]:
@torch.no_grad()
def validation_components(model, loader, thresholds=None):
    model.eval()
    thresholds = np.full(len(DEFECTS), 0.5) if thresholds is None else np.asarray(thresholds)
    positive_intersection = np.zeros(len(DEFECTS), dtype=np.float64)
    positive_denominator = np.zeros(len(DEFECTS), dtype=np.float64)
    positive_pairs = np.zeros(len(DEFECTS), dtype=np.int64)
    negative_pairs = np.zeros(len(DEFECTS), dtype=np.int64)
    correct_empty = np.zeros(len(DEFECTS), dtype=np.int64)
    for images, targets in loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        probabilities = torch.sigmoid(model(images))
        for k, threshold in enumerate(thresholds):
            predicted = probabilities[:, k] >= float(threshold)
            truth = targets[:, k] >= 0.5
            has_truth = truth.flatten(1).any(dim=1)
            if has_truth.any():
                p, y = predicted[has_truth], truth[has_truth]
                positive_intersection[k] += (p & y).sum().item()
                positive_denominator[k] += p.sum().item() + y.sum().item()
                positive_pairs[k] += int(has_truth.sum().item())
            no_truth = ~has_truth
            if no_truth.any():
                negative_pairs[k] += int(no_truth.sum().item())
                correct_empty[k] += int((~predicted[no_truth].flatten(1).any(dim=1)).sum().item())
    fg_dice = np.divide(2 * positive_intersection, positive_denominator,
                        out=np.zeros_like(positive_intersection), where=positive_denominator > 0)
    empty_accuracy = np.divide(correct_empty, negative_pairs, out=np.full(len(DEFECTS), np.nan),
                               where=negative_pairs > 0)
    false_positive_rate = 1.0 - empty_accuracy
    return {
        "foreground_dice": fg_dice,
        "positive_pairs": positive_pairs,
        "empty_accuracy": empty_accuracy,
        "false_positive_pair_rate": false_positive_rate,
        "negative_pairs": negative_pairs,
    }

def print_validation_report(metrics):
    for k, defect in enumerate(DEFECTS):
        print(
            f"{defect}: foreground Dice={metrics['foreground_dice'][k]:.4f} "
            f"(positive pairs={metrics['positive_pairs'][k]}), "
            f"empty accuracy={metrics['empty_accuracy'][k]:.4f} "
            f"(negative pairs={metrics['negative_pairs'][k]})"
        )


In [9]:
EPOCHS = 25
PATIENCE = 6
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT = OUTPUT_DIR / "best_unet.pth"
model = UNet().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
best_fg_dice = -1.0
epochs_without_improvement = 0

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for images, targets in train_loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = segmentation_loss(model(images), targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()
    train_loss /= max(1, len(train_loader))

    metrics = validation_components(model, val_loader)
    observed_classes = metrics["positive_pairs"] > 0
    if not observed_classes.any():
        raise ValueError("Validation split contains no positive anomaly masks.")
    score = float(np.mean(metrics["foreground_dice"][observed_classes]))
    scheduler.step(score)
    print(f"Epoch {epoch + 1}: train_loss={train_loss:.4f}, validation foreground Dice={score:.4f}")
    print_validation_report(metrics)
    if score > best_fg_dice:
        best_fg_dice = score
        epochs_without_improvement = 0
        torch.save(model.state_dict(), CHECKPOINT)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print("Early stopping.")
            break

if not CHECKPOINT.is_file():
    raise RuntimeError("Training ended without saving a model checkpoint.")
model.load_state_dict(torch.load(CHECKPOINT, map_location=DEVICE))
print("Best validation foreground Dice:", best_fg_dice)


Epoch 1: train_loss=1.1783, validation foreground Dice=0.0336
defect_a: foreground Dice=0.0000 (positive pairs=162), empty accuracy=1.0000 (negative pairs=1547)
defect_b: foreground Dice=0.0000 (positive pairs=48), empty accuracy=1.0000 (negative pairs=1661)
defect_c: foreground Dice=0.1344 (positive pairs=615), empty accuracy=0.7002 (negative pairs=1094)
defect_d: foreground Dice=0.0000 (positive pairs=256), empty accuracy=1.0000 (negative pairs=1453)
Epoch 2: train_loss=1.0034, validation foreground Dice=0.2308
defect_a: foreground Dice=0.0000 (positive pairs=162), empty accuracy=1.0000 (negative pairs=1547)
defect_b: foreground Dice=0.0000 (positive pairs=48), empty accuracy=1.0000 (negative pairs=1661)
defect_c: foreground Dice=0.4776 (positive pairs=615), empty accuracy=0.2349 (negative pairs=1094)
defect_d: foreground Dice=0.4455 (positive pairs=256), empty accuracy=0.3985 (negative pairs=1453)
Epoch 3: train_loss=0.7375, validation foreground Dice=0.1209
defect_a: foreground Dic

In [10]:
# Calibrate category thresholds in one validation forward pass (rather than rerunning
# the network for every candidate). This 70/30 proxy prioritizes foreground Dice
# and empty predictions; it is not the official score, whose exact formulas were
# not included in the competition description.
@torch.no_grad()
def calibrate_thresholds(model, loader):
    candidates = np.arange(0.10, 0.951, 0.05)
    n_thresholds = len(candidates)
    intersections = np.zeros((len(DEFECTS), n_thresholds), dtype=np.float64)
    denominators = np.zeros_like(intersections)
    positive_pairs = np.zeros_like(intersections)
    negative_pairs = np.zeros_like(intersections)
    correct_empty = np.zeros_like(intersections)

    model.eval()
    for images, targets in loader:
        probabilities = torch.sigmoid(model(images.to(DEVICE))).cpu().numpy()
        truth_batch = targets.numpy() >= 0.5
        for k in range(len(DEFECTS)):
            for j, threshold in enumerate(candidates):
                predicted = probabilities[:, k] >= threshold
                truth = truth_batch[:, k]
                has_truth = truth.reshape(len(truth), -1).any(axis=1)
                if has_truth.any():
                    pred_positive = predicted[has_truth]
                    true_positive = truth[has_truth]
                    intersections[k, j] += np.logical_and(pred_positive, true_positive).sum()
                    denominators[k, j] += pred_positive.sum() + true_positive.sum()
                    positive_pairs[k, j] += has_truth.sum()
                no_truth = ~has_truth
                if no_truth.any():
                    negative_pairs[k, j] += no_truth.sum()
                    empty = ~predicted[no_truth].reshape(no_truth.sum(), -1).any(axis=1)
                    correct_empty[k, j] += empty.sum()

    fg_dice = np.divide(2 * intersections, denominators, out=np.zeros_like(intersections),
                        where=denominators > 0)
    empty_accuracy = np.divide(correct_empty, negative_pairs,
                               out=np.full_like(correct_empty, np.nan), where=negative_pairs > 0)
    best_thresholds = np.full(len(DEFECTS), 0.5, dtype=np.float32)
    diagnostics = []
    for k, defect in enumerate(DEFECTS):
        if positive_pairs[k].max() == 0:
            best = int(np.argmin(np.abs(candidates - 0.5)))
        else:
            specificity = np.nan_to_num(empty_accuracy[k], nan=0.0)
            scores = 0.7 * fg_dice[k] + 0.3 * specificity
            best = int(np.argmax(scores))
        best_thresholds[k] = candidates[best]
        proxy = 0.7 * fg_dice[k, best] + 0.3 * np.nan_to_num(empty_accuracy[k, best], nan=0.0)
        diagnostics.append((defect, float(candidates[best]), float(fg_dice[k, best]),
                            float(empty_accuracy[k, best]), float(proxy)))
    return best_thresholds, diagnostics

thresholds, calibration_report = calibrate_thresholds(model, val_loader)
for defect, threshold, fg, empty, proxy in calibration_report:
    print(f"{defect}: threshold={threshold:.2f}, foreground Dice={fg:.4f}, "
          f"empty accuracy={empty:.4f}, validation proxy={proxy:.4f}")
print_validation_report(validation_components(model, val_loader, thresholds))


defect_a: threshold=0.10, foreground Dice=0.0000, empty accuracy=1.0000, validation proxy=0.3000
defect_b: threshold=0.10, foreground Dice=0.0000, empty accuracy=1.0000, validation proxy=0.3000
defect_c: threshold=0.35, foreground Dice=0.4972, empty accuracy=0.2011, validation proxy=0.4083
defect_d: threshold=0.55, foreground Dice=0.4288, empty accuracy=0.4425, validation proxy=0.4329
defect_a: foreground Dice=0.0000 (positive pairs=162), empty accuracy=1.0000 (negative pairs=1547)
defect_b: foreground Dice=0.0000 (positive pairs=48), empty accuracy=1.0000 (negative pairs=1661)
defect_c: foreground Dice=0.4972 (positive pairs=615), empty accuracy=0.2011 (negative pairs=1094)
defect_d: foreground Dice=0.4288 (positive pairs=256), empty accuracy=0.4425 (negative pairs=1453)


In [11]:
def locate_test_images(root, requested_ids):
    candidates = [root / "test", root / "test_images"]
    for directory in candidates:
        if directory.is_dir():
            paths = index_images(directory)
            if set(requested_ids).issubset(paths):
                return paths
    # Fallback for datasets with a different test-folder name.
    found = {}
    requested = set(requested_ids)
    valid = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
    for path in root.rglob("*"):
        if path.is_file() and path.suffix.lower() in valid and path.stem in requested:
            if path.stem in found and found[path.stem] != path:
                raise ValueError(f"Multiple files found for test image ID {path.stem!r}")
            found[path.stem] = path
    missing = sorted(requested - set(found))
    if missing:
        raise FileNotFoundError(f"Could not find {len(missing)} test images; examples: {missing[:5]}")
    return found

class TestImageDataset(Dataset):
    def __init__(self, ids, paths):
        self.ids = list(ids)
        self.paths = paths

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        image_id = self.ids[index]
        with Image.open(self.paths[image_id]) as image_file:
            image = np.asarray(image_file.convert("L"), dtype=np.float32) / 255.0
        if image.shape != (HEIGHT, WIDTH):
            raise ValueError(f"Expected image {image_id} to be {(HEIGHT, WIDTH)}, got {image.shape}")
        return torch.from_numpy(np.ascontiguousarray(image[None, ...])), image_id

@torch.no_grad()
def make_submission(model, sample, root, thresholds, output_path):
    # Reuse sample_submission instead of rebuilding IDs or column order.
    image_ids = sample["image_id"].astype(str).drop_duplicates().tolist()
    unknown = set(sample["defect_type"].unique()) - set(DEFECTS)
    if unknown:
        raise ValueError(f"Sample submission has unknown categories: {sorted(unknown)}")
    sample_pairs = list(zip(sample["image_id"].astype(str), sample["defect_type"].astype(str)))
    if len(sample_pairs) != len(set(sample_pairs)):
        raise ValueError("sample_submission.csv contains duplicate image/category pairs")
    expected_pairs = {(image_id, defect) for image_id in image_ids for defect in DEFECTS}
    if set(sample_pairs) != expected_pairs:
        raise ValueError("sample_submission.csv must contain every image x defect_type pair exactly once")
    test_paths = locate_test_images(root, image_ids)
    test_loader = DataLoader(TestImageDataset(image_ids, test_paths), batch_size=BATCH_SIZE,
                             shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
    model.eval()
    prediction_by_pair = {}
    for images, batch_ids in test_loader:
        probabilities = torch.sigmoid(model(images.to(DEVICE))).cpu().numpy()
        for image_index, image_id in enumerate(batch_ids):
            for category_index, defect in enumerate(DEFECTS):
                binary_mask = probabilities[image_index, category_index] >= thresholds[category_index]
                prediction_by_pair[(str(image_id), defect)] = rle_encode(binary_mask)

    result = sample.copy()
    result["image_id"] = result["image_id"].astype(str)
    result["mask_rle"] = [prediction_by_pair.get((image_id, defect), None)
                          for image_id, defect in zip(result["image_id"], result["defect_type"])]
    if result["mask_rle"].isna().any():
        missing_pairs = result.loc[result["mask_rle"].isna(), ["image_id", "defect_type"]].head()
        raise ValueError(f"Missing predictions for sample rows:\n{missing_pairs}")
    if result.duplicated(["image_id", "defect_type"]).any():
        raise ValueError("sample_submission.csv contains duplicate image/category pairs")
    if list(result.columns) != list(sample.columns) or len(result) != len(sample):
        raise AssertionError("Submission structure differs from sample_submission.csv")
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(output_path, index=False)

    # Read back and verify the exact schema and pair coverage.
    checked = pd.read_csv(output_path, keep_default_na=False,
                           dtype={"image_id": str, "defect_type": str, "mask_rle": str})
    if list(checked.columns) != list(sample.columns) or len(checked) != len(sample):
        raise AssertionError("Written CSV failed schema/row-count validation")
    if checked["mask_rle"].isna().any():
        raise AssertionError("Written CSV contains missing mask_rle values")
    expected_pairs = set(zip(sample["image_id"].astype(str), sample["defect_type"].astype(str)))
    actual_pairs = set(zip(checked["image_id"].astype(str), checked["defect_type"].astype(str)))
    if actual_pairs != expected_pairs:
        raise AssertionError("Written CSV does not cover the sample image/category pairs")
    return output_path

submission_path = make_submission(
    model, sample_submission, ROOT, thresholds, OUTPUT_DIR / "submission.csv"
)
print("Submission saved and validated:", submission_path)
print(pd.read_csv(submission_path, keep_default_na=False).head())


Submission saved and validated: /kaggle/working/submission.csv
   ID    image_id defect_type mask_rle
0   1  WAFNKCRR7Q    defect_a         
1   2  WAFNKCRR7Q    defect_b         
2   3  WAFNKCRR7Q    defect_c         
3   4  WAFNKCRR7Q    defect_d         
4   5  YZXKEWSUDD    defect_a         
